# Generate an Evaluation Dataset for a Report Generation Agent

In this notebook, you'll create a synthetic evaluation dataset using **NVIDIA NeMo Data Designer** for the **Report Generation Agent** from Module 1, using your learnings from the previous evaluation data notebook.

## 1. Setup

First, let's install the Data Designer library and set up our environment.

In [1]:
# Install Data Designer if needed
%pip install data-designer -q

# Load environment variables
from dotenv import load_dotenv
_ = load_dotenv("../../variables.env")
_ = load_dotenv("../../secrets.env")

import os
import json
from pathlib import Path

print("Completed setup")

Note: you may need to restart the kernel to use updated packages.
Completed setup


## 2. Configure DataDesigner

Initialize Data Designer with default settings.

In [5]:
from data_designer.essentials import (
    DataDesigner,
    DataDesignerConfigBuilder,
    LLMStructuredColumnConfig,
)

# Initialize Data Designer with default settings
data_designer = DataDesigner()
config_builder = DataDesignerConfigBuilder()

print("Completed initialization")

Completed initialization


## 3. Create Seed Data with Report Topics

Unlike the RAG agent which retrieves from a knowledge base, the report agent generates comprehensive reports on given topics. This will change the type of evaluation metrics we use, and require a different type of evaluation data.

Let's start with some seed data. We'll start a list of manually curated topics. This will help avoid semantic duplication, ensuring that the evaluation data covers a variety of different domains.

In [6]:
import pandas as pd

# Define diverse report topics across different domains
report_topics_data = [
    {
        "domain": "Technology",
        "topic_area": "Artificial Intelligence",
    },
    {
        "domain": "Energy",
        "topic_area": "Renewable Energy"
    },
    {
        "domain": "Business",
        "topic_area": "Remote Work",
    },
    {
        "domain": "Technology",
        "topic_area": "Quantum Computing",
    },
    {
        "domain": "Supply Chain",
        "topic_area": "Sustainability",
    },
    {
        "domain": "Cybersecurity",
        "topic_area": "Financial Sector Threats",
    },
    {
        "domain": "Healthcare",
        "topic_area": "Digital Health Technologies",
    },
    {
        "domain": "Education",
        "topic_area": "Online Learning Platforms",
    },
    {
        "domain": "Climate",
        "topic_area": "Carbon Capture Technologies",
    },
    {
        "domain": "Finance",
        "topic_area": "Cryptocurrency Regulation"
    }
]

# Create seed data and save it for Data Designer
seed_data = pd.DataFrame(report_topics_data)
seed_file_path = Path("../../data/evaluation/report_seed_data.parquet")

seed_ref = data_designer.make_seed_reference_from_dataframe(
    dataframe=seed_data,
    file_path=seed_file_path
)

print(f"Created seed data with {len(seed_data)} topics")

[17:50:53] [INFO] 💾 Saving seed dataset to ../../data/evaluation/report_seed_data.parquet


Created seed data with 10 topics


## 4. Configure Data Designer for Report Evaluation

Now we'll configure Data Designer to generate complete evaluation test cases using **LLM-Structured Columns**. This approach is different from the text-based columns we used for the RAG agent—instead of generating separate text fields, we use a Pydantic model to define the entire structure we want:

- **Topic** - A specific professional report topic
- **Expected Sections** - A list of 6-8 section titles  
- **Quality Criteria** - A nested object with `should_include` and `should_avoid` lists

Using `LLMStructuredColumnConfig`, Data Designer generates the topic, sections, and criteria together in one LLM call, ensuring coherence. It also ensures all required fields are present and correctly typed.


In [7]:
from typing import List
from pydantic import BaseModel, Field

# Define Pydantic models for structured output
class QualityCriteria(BaseModel):
    should_include: List[str] = Field(
        description="3-4 specific things that SHOULD be included for a high-quality report"
    )
    should_avoid: List[str] = Field(
        description="3-4 things that SHOULD be avoided to maintain quality and objectivity"
    )

class ReportEvaluationCase(BaseModel):
    topic: str = Field(
        description="Specific, professional report topic"
    )
    expected_sections: List[str] = Field(
        description="6-8 section titles for a comprehensive professional report"
    )
    quality_criteria: QualityCriteria = Field(
        description="Quality criteria for evaluating the report"
    )

# Create config builder with seed data
config_builder = DataDesignerConfigBuilder()
config_builder.with_seed_dataset(seed_ref)

# Generate complete structured evaluation case in a single column
config_builder.add_column(
    LLMStructuredColumnConfig(
        name="evaluation_case",
        model_alias="nvidia-text",
        output_format=ReportEvaluationCase,
        prompt="""Generate a complete evaluation test case for a professional report.

Context:
- Domain: {{ domain }}
- Topic Area: {{ topic_area }}

Create an evaluation case that includes:

1. TOPIC: A specific, professional report topic that is:
   - Focused and not too broad
   - Relevant to current trends in {{ topic_area }}
   - Suitable for a comprehensive multi-section research report

2. EXPECTED_SECTIONS: 6-8 section titles that:
   - Start with an introduction or executive summary
   - Include substantive middle sections covering key aspects
   - End with conclusions, recommendations, or future outlook
   - Use clear, professional section titles
   - Follow a logical flow

3. QUALITY_CRITERIA:
   - should_include: 3-4 specific things that SHOULD be included for quality
   - should_avoid: 3-4 things that SHOULD be avoided to maintain objectivity""",
    )
)

print("Dataset configuration built with structured output")

Dataset configuration built with structured output


## 5. Preview the Data Structure

Let's preview what the generated test cases will look like.

In [8]:
# Preview a few sample records to see what will be generated
preview = data_designer.preview(config_builder=config_builder, num_records=3)
preview.display_sample_record()

[17:51:01] [INFO] 👀 Preview generation in progress
[17:51:01] [INFO] ✅ Validation passed
[17:51:01] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[17:51:01] [INFO] 🩺 Running health checks for models...
[17:51:01] [INFO]   |-- 👀 Checking 'nvidia/nvidia-nemotron-nano-9b-v2' in provider named 'nvidia' for model alias 'nvidia-text'...
[17:51:03] [INFO]   |-- ✅ Passed!
[17:51:03] [INFO] 🌱 Sampling 3 records from seed dataset
[17:51:03] [INFO]   |-- seed dataset size: 10 records
[17:51:03] [INFO]   |-- sampling strategy: ordered
[17:51:03] [INFO] 🗂️ Preparing llm-structured column generation
[17:51:03] [INFO]   |-- column name: 'evaluation_case'
[17:51:03] [INFO]   |-- model config:
{
    "alias": "nvidia-text",
    "model": "nvidia/nvidia-nemotron-nano-9b-v2",
    "inference_parameters": {
        "temperature": 0.85,
        "top_p": 0.95,
        "max_tokens": null,
        "max_parallel_requests": 4,
        "timeout": null,
        "extra_body": null
    },
    "provide

                                                   Seed Columns                                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                                ┃ Value                                                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ domain                              │ Technology                                                                │
├─────────────────────────────────────┼───────────────────────────────────────────────────────────────────────────┤
│ topic_area                          │ Artificial Intelligence                                                   │
└─────────────────────────────────────┴───────────────────────────────────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                 Generated Columns                                                 
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name            ┃ Value                                                                                         ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ evaluation_case │ {                                                                                             │
│                 │     'topic': 'Ethical Implications of Generative AI in Content Creation',                     │
│                 │     'expected_sections': [                                                                    │
│                 │         'Executive Summary',                                                                  │
│                 │         'Methodology and Scope',                                                              │
│                 │         'Technical Overview of Generative AI Models',                                         │
│                 │         'Ethical Challenges in Content Generation',                                           │
│                 │         'Case Studies: AI-Generated Content in Media and Marketing',                          │
│                 │         'Policy Recommendations for Ethical AI Deployment',                                   │
│                 │         'Future Outlook and Risk Mitigation Strategies'                                       │
│                 │     ],                                                                                        │
│                 │     'quality_criteria': {                                                                     │
│                 │         'should_include': [                                                                   │
│                 │             'A thorough analysis of existing ethical frameworks (e.g., transparency, bias,    │
│                 │ accountability) relevant to generative AI',                                                   │
│                 │             'Real-world examples of AI-generated content misuse or success',                  │
│                 │             'Data-driven evaluation of bias or misinformation risks in AI outputs',           │
│                 │             'Balanced discussion of both opportunities and risks for stakeholders'            │
│                 │         ],                                                                                    │
│                 │         'should_avoid': [                                                                     │
│                 │             'Overly technical jargon

## 6. Generate the Full Dataset

Now generate the complete evaluation dataset. We'll generate one test case per topic seed to ensure diverse coverage.

In [9]:
# Generate evaluation data (one test case per topic seed)
num_topics = len(report_topics_data)

dataset_results = data_designer.create(
    config_builder=config_builder,
    num_records=num_topics,
    dataset_name="report_agent_test_cases"
)

# Convert to list of dictionaries for processing
eval_data = dataset_results.load_dataset().to_dict('records')

print(f"\nGenerated {len(eval_data)} test cases")

[17:51:23] [INFO] 🎨 Creating Data Designer dataset
[17:51:23] [INFO] ✅ Validation passed
[17:51:23] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[17:51:23] [INFO] 📂 Dataset path '/home/nvidia/workshop-build-an-agent/code/3-agent-evaluation/artifacts/report_agent_test_cases' already exists. Dataset from this session
		     will be saved to '/home/nvidia/workshop-build-an-agent/code/3-agent-evaluation/artifacts/report_agent_test_cases_12-11-2025_175123' instead.
[17:51:23] [INFO] 🩺 Running health checks for models...
[17:51:23] [INFO]   |-- 👀 Checking 'nvidia/nvidia-nemotron-nano-9b-v2' in provider named 'nvidia' for model alias 'nvidia-text'...
[17:51:24] [INFO]   |-- ✅ Passed!
[17:51:24] [INFO] ⏳ Processing batch 1 of 1
[17:51:24] [INFO] 🌱 Sampling 10 records from seed dataset
[17:51:24] [INFO]   |-- seed dataset size: 10 records
[17:51:24] [INFO]   |-- sampling strategy: ordered
[17:51:24] [INFO] 🗂️ Preparing llm-structured column generation
[17:51:24] [INFO]   |-- c


Generated 10 test cases


## 7. Save the Test Data

You've successfully generated synthetic test data for evaluating your Report Generation agent!

Just like before, let's clean up and save the generated data in a format ready for evaluation. 

In [ ]:
# Set output path
output_path = Path("../../data/evaluation/synthetic_report_agent_test_cases.json")
formatted_data = []

for record in eval_data:
    # Extract evaluation_case which contains our structured data
    eval_case = record.get("evaluation_case", {})
    
    # Handle both dict and Pydantic model formats
    if hasattr(eval_case, 'model_dump'):
        eval_case = eval_case.model_dump()
    
    # Extract quality criteria
    quality_criteria = eval_case.get("quality_criteria", {})
    if hasattr(quality_criteria, 'model_dump'):
        quality_criteria = quality_criteria.model_dump()
    
    formatted_data.append({
        "topic": eval_case.get("topic", ""),
        "expected_sections": eval_case.get("expected_sections", []),
        "quality_criteria": {
            "should_include": quality_criteria.get("should_include", []),
            "should_avoid": quality_criteria.get("should_avoid", [])
        }
    })

# Save to JSON file
with open(output_path, 'w') as f:
    json.dump(formatted_data, f, indent=2)

print(f"Saved {len(formatted_data)} test cases to {output_path.absolute()}")

Saved 10 test cases to /home/nvidia/workshop-build-an-agent/code/3-agent-evaluation/../../data/evaluation/synthetic_report_agent_test_cases.json


## What's Next?

Check out <button onclick="openOrCreateFileInJupyterLab('data/evaluation/synthetic_rag_agent_test_cases.json');"><i class="fa-brands fa-python"></i> synthetic_report_agent_test_cases.json </button> to view your generated data.

You've built evaluation datasets for both of your agents. Now, you're ready to move on to the next step- putting together an evaluation pipeline and using the data you've generated to evaluate agent performance.